# Order Management System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivikasavnish/algo-trading-notebooks/blob/main/notebooks/16_order_management_system.ipynb)

Place, modify, and cancel orders through a ServLoci-proxied session.

Part 16 of 24 in the [ServLoci algo/options trading notebook series](https://comm.servloci.in/docs) — full index in `notebooks/README.md`.

## Setup

In [ ]:
# Get your dedicated static IPv6 + SOCKS5 credentials free:
#   https://comm.servloci.in/register        (or /auth/google?free=1 for an instant trial)
# Your api_key / api_secret pair shows up in the portal after signup:
#   https://comm.servloci.in/user
!pip install -q "requests[socks]"
!curl -sL https://comm.servloci.in/sdk/servloci.py -o servloci.py

import os
from servloci import ServLoci

SERVLOCI_API_KEY = os.environ.get("SERVLOCI_API_KEY", "dhan:1000000001")   # broker:client_id
SERVLOCI_API_SECRET = os.environ.get("SERVLOCI_API_SECRET", "")            # from the portal — leave blank to run this notebook in demo mode

sl = None
if SERVLOCI_API_SECRET:
    sl = ServLoci(api_key=SERVLOCI_API_KEY, api_secret=SERVLOCI_API_SECRET)
    print("ServLoci configured:", sl.host, sl.port)
else:
    print("SERVLOCI_API_SECRET not set — running in demo mode (no live proxy calls).")

A thin OMS wrapper over a broker session routed through the static IP — plug in your broker's REST base URL and auth headers.

In [ ]:
class OrderManager:
    """Place / modify / cancel orders through a ServLoci-proxied session."""
    def __init__(self, session, base_url):
        self.session = session
        self.base_url = base_url.rstrip("/")

    def place(self, order: dict) -> dict:
        r = self.session.post(f"{self.base_url}/orders", json=order, timeout=8)
        r.raise_for_status()
        return r.json()

    def modify(self, order_id: str, changes: dict) -> dict:
        r = self.session.put(f"{self.base_url}/orders/{order_id}", json=changes, timeout=8)
        r.raise_for_status()
        return r.json()

    def cancel(self, order_id: str) -> dict:
        r = self.session.delete(f"{self.base_url}/orders/{order_id}", timeout=8)
        r.raise_for_status()
        return r.json()

if sl:
    oms = OrderManager(sl.session(), base_url=os.environ.get("BROKER_API_BASE", "https://api.example-broker.com/v1"))
    order = {"symbol": "NIFTY24800CE", "transaction_type": "BUY", "quantity": 75, "order_type": "MARKET", "product": "INTRADAY"}
    # resp = oms.place(order)   # uncomment once BROKER_API_BASE + broker auth headers are set
    print("OMS ready — set BROKER_API_BASE and auth headers, then uncomment oms.place(order)")
else:
    print("Demo mode — set SERVLOCI_API_SECRET above to build a live OrderManager.")

---

« Previous: [Position Sizing & Risk Management](15_position_sizing_risk_management.ipynb)  
Next: [Paper Trading Loop](17_paper_trading_loop.ipynb) »

Try the concepts above interactively: [Options Strategy Builder](https://comm.servloci.in/tools/strategy-builder) · [Docs](https://comm.servloci.in/docs) · [Get your static IP](https://comm.servloci.in/register)